In [1]:
import json
from pathlib import Path

import numpy as np

from pyscf import gto

import sys

In [2]:
import yaml
dict_ = {"molecule": [], "spin": {}, "charge": {}}
BASIS = "def2-svp"
dataset = "s66x8"
dict_[f"molecule_{dataset}"] = []
dict_[f"reaction-{dataset}"] = {}

data_path = Path("./sets")

# read s66x8.yaml
class RubyTagSafeLoader(yaml.SafeLoader):
    pass


def _ignore_unknown_tags(loader, tag_suffix, node):
    if isinstance(node, yaml.MappingNode):
        return loader.construct_mapping(node, deep=True)
    if isinstance(node, yaml.SequenceNode):
        return loader.construct_sequence(node, deep=True)
    return loader.construct_scalar(node)


RubyTagSafeLoader.add_multi_constructor("!", _ignore_unknown_tags)

with open(data_path / "s66x8.yaml", "r") as f:
    s66x8 = yaml.load(f, Loader=RubyTagSafeLoader)

with open(data_path / "s66.yaml", "r") as f:
    s66 = yaml.load(f, Loader=RubyTagSafeLoader)

print(s66["items"])

[{'name': '01 Water ... Water', 'shortname': '01_Water-Water', 'geometry': 'S66:01', 'reference_value': -5.011, 'setup': {}, 'group': 'H-bonds', 'tags': '1 H-bond'}, {'name': '02 Water ... MeOH', 'shortname': '02_Water-MeOH', 'geometry': 'S66:02', 'reference_value': -5.701, 'setup': {}, 'group': 'H-bonds', 'tags': '1 H-bond'}, {'name': '03 Water ... MeNH2', 'shortname': '03_Water-MeNH2', 'geometry': 'S66:03', 'reference_value': -7.036, 'setup': {}, 'group': 'H-bonds', 'tags': '1 H-bond'}, {'name': '04 Water ... Peptide', 'shortname': '04_Water-Peptide', 'geometry': 'S66:04', 'reference_value': -8.22, 'setup': {}, 'group': 'H-bonds', 'tags': '1 H-bond'}, {'name': '05 MeOH ... MeOH', 'shortname': '05_MeOH-MeOH', 'geometry': 'S66:05', 'reference_value': -5.851, 'setup': {}, 'group': 'H-bonds', 'tags': '1 H-bond'}, {'name': '06 MeOH ... MeNH2', 'shortname': '06_MeOH-MeNH2', 'geometry': 'S66:06', 'reference_value': -7.666, 'setup': {}, 'group': 'H-bonds', 'tags': '1 H-bond'}, {'name': '07 M

In [3]:
curve_geom_dict = {}

for items in s66x8["items"]:
    xyz_file = (Path("sets/S66x8") / f"{items['shortname']}.xyz").resolve().as_posix()
    mol = gto.M(
        atom=xyz_file,
        basis=BASIS,
        verbose=0,
        charge=0,
        spin=0,
        unit="B",
    )
    mol.build()

    molecule = []
    for i_atom in mol._atom:
        molecule.append(
            [
                i_atom[0],
                i_atom[1][0],
                i_atom[1][1],
                i_atom[1][2],
            ]
        )

    dict_i_name = f"{dataset}-{items["curve"]}-{items["curve_x"]}"
    dict_[dict_i_name] = molecule
    dict_[f"molecule_{dataset}"].append(dict_i_name)
    dict_["molecule"].append(dict_i_name)
    dict_["charge"][dict_i_name] = mol.charge
    dict_["spin"][dict_i_name] = mol.spin

    if items["curve"] not in curve_geom_dict:
        curve_geom_dict[items["curve"]] = [molecule]
    else:
        curve_geom_dict[items["curve"]].append(molecule)

for key, value in curve_geom_dict.items():
    molecule1 = value[1]
    molecule0 = value[0]
    mol1 = []
    mol2 = []
    for i, atom in enumerate(molecule1):
        distance = np.linalg.norm(np.array(atom[1:]) - np.array(molecule0[i][1:]))
        if distance < 1e-5:
            mol1.append(atom)
        else:
            mol2.append(atom)

    dict_i_name = f"{dataset}-{key}-a"
    dict_[dict_i_name] = mol1
    dict_[f"molecule_{dataset}"].append(dict_i_name)
    dict_["molecule"].append(dict_i_name)
    dict_["charge"][dict_i_name] = 0
    dict_["spin"][dict_i_name] = 0

    dict_i_name = f"{dataset}-{key}-b"
    dict_[dict_i_name] = mol2
    dict_[f"molecule_{dataset}"].append(dict_i_name)
    dict_["molecule"].append(dict_i_name)
    dict_["charge"][dict_i_name] = 0
    dict_["spin"][dict_i_name] = 0


for items in s66x8["items"]:
    i_reaction = f"{items["curve"]}_{items["curve_x"]}"
    systems = [
        f"{dataset}-{items['curve']}-{items['curve_x']}",
        f"{dataset}-{items['curve']}-a",
        f"{dataset}-{items['curve']}-b",
    ]
    stoichiometry = [1, -1, -1]
    reference = items["reference_value"]
    dict_[f"reaction-{dataset}"][i_reaction] = {
        "systems": systems,
        "stoichiometry": stoichiometry,
        "reference": reference,
    }
        

In [4]:
with open(f"gmtkn-{dataset}-{BASIS}.json", "w") as f:
    json.dump(dict_, f)

In [5]:
import json

BASIS = "def2-svp"
dataset = "s66x8"

with open(f"gmtkn-{dataset}-{BASIS}.json", "r") as f:
    dict_ = json.load(f)

for subset in ["ab", "a", "b"]:
    dict_[f"reaction-molecule_{subset}"] = {}

for key, value in dict_[f"reaction-{dataset}"].items():
    distance = key.split("_")[-1]
    if f"reaction-molecule_{distance}" not in dict_:
        dict_[f"reaction-molecule_{distance}"] = {}
    dict_[f"reaction-molecule_{distance}"][key] = value


for key in dict_[f"molecule"]:
    distance = key.split("-")[-1]
    if f"molecule_{distance}" not in dict_:
        dict_[f"molecule_{distance}"] = []
    dict_[f"molecule_{distance}"].append(key)

with open(f"gmtkn-{dataset}-{BASIS}.json", "w") as f:
    json.dump(dict_, f)